Taitanic : Machine Learning from Disater


Survived Prediction


The goal : This is a kernel for study. The goal is an accuracy of 0.9

# <a id='0'>Content</a>
- <a href='#1'>Introduction</a>  
- <a href='#2'>Prepare the data analysis</a>  
- <a href='#3'>Data exploration</a>   
 - <a href='#31'>Check the data</a>   
 - <a href='#32'>Density plots of features</a>   
 - <a href='#33'>Distribution of mean and std</a>   
 - <a href='#34'>Distribution of min and max</a>   
  - <a href='#35'>Distribution of skew and kurtosis</a>   
 - <a href='#36'>Features correlations</a>   
 - <a href='#37'>Duplicate values</a>   
- <a href='#4'>Feature engineering</a>
- <a href='#5'>Model</a>
- <a href='#6'>Submission</a>  
- <a href='#7'>References</a>

# <a id='1'>Introduction</a>  
#### **Competition background**
The sinking of the RMS Titanic is one of the most infamous shipwrecks in history.  On April 15, 1912, during her maiden voyage, the Titanic sank after colliding with an iceberg, killing 1502 out of 2224 passengers and crew. This sensational tragedy shocked the international community and led to better safety regulations for ships.

One of the reasons that the shipwreck led to such loss of life was that there were not enough lifeboats for the passengers and crew. Although there was some element of luck involved in surviving the sinking, some groups of people were more likely to survive than others, such as women, children, and the upper-class.

In this challenge, we ask you to complete the analysis of what sorts of people were likely to survive. In particular, we ask you to apply the tools of machine learning to predict which passengers survived the tragedy.

#### **Evaluation**


#### **Timeline**


**Data**

train.csv, test.csv, sample_submission.csv

# <a id='2'>Prepare for data analysis</a>  


## Load packages

In [ ]:
import warnings
warnings.filterwarnings('ignore')


import scipy as sp
import pandas as pd
import re

import matplotlib as mlt
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

## Load data   

Let's check what data files are available.

In [ ]:
train = pd.read_csv('../input/train.csv')
test = pd.read_csv('../input/test.csv')

# <a id='3'>Data exploration</a>  

## <a id='31'>Check the data</a>  

Let's check the train and test set.

In [ ]:
# Data shape
print(train.shape)
print(test.shape)

# Check Columns
print(train.columns)
print(test.columns)

# Column comparison
print(set(train.columns)-set(test.columns))
print(set(test.columns)-set(train.columns))

Train data is approximately twice as large as test data.

Columns are the same except for target variables.

Let's glimpse train and test dataset.

In [ ]:
train.head(3)

In [ ]:
test.head(3)

Train contains:  

* **PassengerId** (string);  
* **Survived**;  
* **2** numerical variables, named **Age** and **Fare**
* **8** categorical variables, named **Pclass**, **Name**, **Sex**, **SibSp**, **Parch**, **Ticket**, **Cabin**, **Embarked**

Test contains:  

* **PassengerId** (string);  
* **2** numerical variables, named **Age** and **Fare**
* **8** categorical variables, named **Pclass**, **Name**, **Sex**, **SibSp**, **Parch**, **Ticket**, **Cabin**, **Embarked**

Let's check if there are any missing data. We will also check the type of data.

We check first train.

In [ ]:
def missing_data(data):
    total = data.isnull().sum()
    percent = (data.isnull().sum()/data.isnull().count()*100)
    types = [str(data[i].dtype) for i in data.columns]
    
    df = pd.DataFrame({'Total':total, 'Precent':percent, 'Types':types})
    
    return(sp.transpose(df))

In [ ]:
missing_data(train)

In [ ]:
missing_data(test)

In [ ]:
train.describe()

In [ ]:
test.describe()

? ? ? ????. skew, std, etc...

missing data? ??? ??????.

????? ?? ?? ?? train? test? ?? ???????.

In [ ]:
Survived = train['Survived']
data = pd.concat([train.drop(columns='Survived'), test])

In [ ]:
missing_data(data)

In [ ]:
data['Age'].fillna(data['Age'].median(), inplace=True)

In [ ]:
data['Fare'].fillna(data['Fare'].median(), inplace=True)

In [ ]:
data['Embarked'].fillna(data['Embarked'].mode()[0], inplace=True)

In [ ]:
missing_data(data)

??? ???? ?? ?? ??? ???. ???? ??? ?? ?? ??? ??? no value?? ?? ????.

In [ ]:
data['Cabin'].fillna('no value', inplace=True)

In [ ]:
missing_data(data)

???? ?? ???.

?? train, test data? ?????.

In [ ]:
train = data.loc[:train.index[-1]]
test = data.loc[train.index[-1]:][1:]
train = pd.concat([Survived, train], axis=1)

## <a id='32'>Density plots of features</a>  
???? ?? ??? ??? ???.

?? target variable? Survived? ?? ???

In [ ]:
sns.countplot(x='Survived', data=train).set_title('Survived probablity density function')
plt.show()

imbalance ? ???? ?? ? ? ????.

????? ??? ?????.

??? ???? ?? ???? ?????.

In [ ]:
features = ['Age', 'Fare']
nrow=1;ncol=2
f, axes = plt.subplots(nrow, ncol, figsize=(20, 5))
for idx, feature in enumerate(features):
    plt.subplot(nrow, ncol, idx+1)
    sns.distplot(data[feature]).set_title('PDF '+str(feature))
plt.show()

Age? ?? ??? ?? ?? ??? ??? ?? ? ? ??. ??? ????? ????? ??? ?? ? ?? ? ??. 25? ???? ??? ????? ????? ???? ??? ????.

Fare? ?? ??? ?? ???? ?? ?? ?? ??, skew?? ??. log? ?? ???? ?? ??? ????? ?? ???? ??? ??? ???? ?? ?? ? ??.

??? ??? ??? ?? ???????.

PassengerId? ??, ?? Id??? ???? ??? ???? ???? ?????.

Name, Ticket, Cabin? ???? ?? ?? ? ?? ??? ?? Feature engineering? ?? ????? ??? ???????.

In [ ]:
features =  ['Pclass', 'Sex', 'SibSp', 'Parch', 'Embarked'] # 'Name', 'Ticket', 'Cabin'
nrow=2;ncol=3
f, axes = plt.subplots(nrow, ncol, figsize=(20, 15))
for idx, feature in enumerate(features):
    plt.subplot(nrow, ncol, idx+1)
    sns.countplot(data[feature])
    plt.xlabel(feature, fontsize=15)
#     plt.xticks(rotation=90)
plt.show()

?? ??? ?? ?? ?? ? ? ????. ??? ???? ??? ?? ? ??? ??? ??? ????.

## <a id='33'>Feature Engineering1</a>  
??? ???? ?? ???? ?? ???? ??? ?????.

??? ???? ??? ?????.

In [ ]:
data['Title'] = data['Name'].apply(lambda x: x.split(', ')[1].split('.')[0])
sns.countplot(data['Title'])
plt.xticks(rotation=-45)
plt.show()

In [ ]:
nrow=1;ncol=2
f, axes = plt.subplots(nrow, ncol, figsize=(15, 5))
plt.subplot(nrow, ncol, 1)
sns.countplot(data['Ticket'].value_counts())
plt.subplot(nrow, ncol, 2)
sns.countplot(data['Cabin'].value_counts())
plt.show()

????? ??? ?????.

y? ??? ??? ??-??, ??-??? ?????.

??-?? ?????.

In [ ]:
features = ['Age', 'Fare']
nrow = 1;ncol=2
f, axes = plt.subplots(nrow, ncol, figsize=(15, 5))
for indx, feature in enumerate(features):
    plt.subplot(nrow, ncol, indx+1)
    sns.distplot(train.loc[Survived==0, feature])
    sns.distplot(train.loc[Survived==1, feature])
plt.show()

In [ ]:
train = data.loc[:train.index[-1]]
test = data.loc[train.index[-1]:][1:]
train = pd.concat([Survived, train], axis=1)

In [ ]:
import sklearn

In [ ]:
le = sklearn.preprocessing.LabelEncoder()
train['Sex_code'] = le.fit_transform(train['Sex'])
train['Ticket_code'] = le.fit_transform(train['Ticket'])
train['Cabin_code'] = le.fit_transform(train['Cabin'])
train['Embarked_code'] = le.fit_transform(train['Embarked'])
train['Title_code'] = le.fit_transform(train['Title'])

In [ ]:
features = ['Survived', 'Pclass', 'Sex_code', 'Age', 'SibSp', 'Parch', 'Ticket_code', 'Fare', 'Cabin_code','Embarked_code', 'Title_code']

In [ ]:
sns.pairplot(train[['Survived', 'Age', 'Fare']], hue='Survived', vars=['Age', 'Fare'])
plt.title('Pair plot')
plt.show()

pari plot?? ?????~ ??? ??? ?? ??? ?? ? ???

In [ ]:
sns.boxplot(x='Survived', y='Age', data=train)

In [ ]:
sns.boxplot(x='Survived', y='Fare', data=train)

In [ ]:
sns.countplot(x='Sex', hue='Survived', data=train)

??? ?? ??? ???? y? ??? ?, ? pair plot??? ??? ??? box plot? ?? ???? ??. ??? ??

?? y? ??, x? ??? ?? ???? ??? ?? ??? ? ? ? ?? ? ??.

violine? ???? ????

??? ??? ? countplot?? ???? ?? ??? hue??


feature engineering and modeling

In [ ]:
le = sklearn.preprocessing.LabelEncoder()
data['Sex_code'] = le.fit_transform(data['Sex'])
data['Ticket_code'] = le.fit_transform(data['Ticket'])
data['Cabin_code'] = le.fit_transform(data['Cabin'])
data['Embarked_code'] = le.fit_transform(data['Embarked'])
data['Title_code'] = le.fit_transform(data['Title'])

In [ ]:
data = data[['Pclass', 'Age', 'SibSp', 'Parch',
        'Fare', 'Sex_code', 'Ticket_code', 'Cabin_code', 'Embarked_code', 'Title_code']]

In [ ]:
train = data.loc[:train.index[-1]]
test = data.loc[train.index[-1]:][1:]
train = pd.concat([Survived, train], axis=1)

In [ ]:
train = train[['Survived', 'Age', 'Fare']]
test = test[['Age', 'Fare']]

In [ ]:
# import sklearn.linear_model
# help(sklearn.linear_model.LogisticRegression)

In [ ]:
model = sklearn.linear_model.LogisticRegression()

In [ ]:
# dir(sklearn.model_selection.train_test_split)

In [ ]:
trn, val, trn_y, val_y = sklearn.model_selection.train_test_split(train[['Age', 'Fare']], Survived)

In [ ]:
model.fit(trn, trn_y)

In [ ]:
# predict = model.predict_proba(val)[:,1]
predict = model.predict(val)

In [ ]:
sklearn.metrics.accuracy_score(val_y, predict)

In [ ]:
sklearn.metrics.f1_score(val_y, predict)

In [ ]:
sklearn.metrics.confusion_matrix(val_y, predict)

In [ ]:
fpr, tpr, threshold = sklearn.metrics.roc_curve(val_y, predict)
roc_auc = sklearn.metrics.auc(fpr, tpr)

In [ ]:
plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = round(roc_auc,2))
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(val_y, predict, target_names=['Survived', 'Not Survived']))

In [ ]:
neigh = sklearn.neighbors.KNeighborsClassifier(n_neighbors=5)
neigh.fit(trn, trn_y)

In [ ]:
predict2 = neigh.predict(val)

In [ ]:
print(sklearn.metrics.classification_report(val_y, predict2, target_names=['Survived', 'Not Survived']))

In [ ]:
predict2 = neigh.predict_proba(val)[:,1]

In [ ]:
fpr, tpr, threshold = sklearn.metrics.roc_curve(val_y, predict2)
roc_auc = sklearn.metrics.auc(fpr, tpr)

In [ ]:
plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = round(roc_auc,2))
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
# help(sklearn.svm.SVC)

clf = sklearn.svm.SVC(random_state=1028, probability=True, kernel='linear')
clf = clf.fit(trn, trn_y)

In [ ]:
predict3 = clf.predict(val)

In [ ]:
# help(sklearn.metrics.classification_report)
print(sklearn.metrics.classification_report(val_y, predict3, target_names=['O', 'X']))

In [ ]:
predict3 = clf.predict_proba(val)[:,1]

In [ ]:
fpr, tpr, threshold = sklearn.metrics.roc_curve(val_y, predict3)
roc_auc = sklearn.metrics.auc(fpr, tpr)

In [ ]:
plt.title('ROC Curve')
plt.plot(fpr, tpr, 'b', label=round(roc_auc, 2))
plt.legend(loc = 'lower right')
plt.plot([0,1], [0,1], 'r--')
plt.xlabel('false positive ratio')
plt.ylabel('true positive ratio')
plt.xlim([0,1]);plt.ylim([0,1])
plt.show()

In [ ]:
# help(sklearn.ensemble.RandomForestClassifier)

clf = sklearn.ensemble.RandomForestClassifier(max_depth=4,
 random_state=1028).fit(trn, trn_y)

In [ ]:
predict4 = clf.predict(val)

In [ ]:
print(sklearn.metrics.classification_report(val_y, predict4))

In [ ]:
predict4 = clf.predict_proba(val)[:,1]

In [ ]:
fpr, tpr, threshold = sklearn.metrics.roc_curve(val_y, predict4)
roc_auc = sklearn.metrics.auc(fpr, tpr)

In [ ]:
plt.title('Roc curve')
plt.plot(fpr, tpr, 'b', label=round(roc_auc, 2))
plt.plot([0,1],[0,1],'r--')
plt.xlim([0,1])
plt.ylim([0,1])
plt.xlabel('false positive ratio')
plt.ylabel('true positive ratio')
plt.legend(loc='lower right')
plt.show()

In [ ]:
feature_imp = pd.DataFrame(clf.feature_importances_, index=['Age', 'Fare'])

In [ ]:
feature_imp = feature_imp[0].sort_values(ascending=False)

In [ ]:
sns.barplot(x=feature_imp, y=feature_imp.index)

In [ ]:
feature_imp = pd.Series(clf.feature_importances_,index=['A','B']).sort_values(ascending=False)
feature_imp

In [ ]:
corr = train.corr()
f, axes = plt.subplots(1, 1, figsize=(3, 3))

plt.subplot(1, 1, 1)
sns.heatmap(corr, cmap='coolwarm', square=True, annot=True, fmt='.2f', annot_kws={"size": 12}, linewidths=.05).set_title('heat map_ ')
plt.show()

In [ ]:
help(sns.heatmap)

In [ ]:
corr = train.drop(columns='id').corr(method='spearman')
fig, axes = plt.subplots(1, 1, figsize=(20,12))

hm = sns.heatmap(corr, 
                 ax=axes,           # Axes in which to draw the plot, otherwise use the currently-active Axes.
                 cmap="coolwarm", # Color Map.
                 square=True,    # If True, set the Axes aspect to “equal” so each cell will be square-shaped.
                 annot=True, 
                 fmt='.2f',       # String formatting code to use when adding annotations.
                 annot_kws={"size": 12},
                 linewidths=.05)

fig.subplots_adjust(top=0.93)
fig.suptitle('Spearman Correlation Heatmap', 
              fontsize=14, 
              fontweight='bold')
plt.show()

Missing value fill

In [ ]:
train['Age'] = train['Age'].fillna(round(np.mean(train['Age'])))
train['Cabin'] = train['Cabin'].fillna('no value')
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].value_counts().index[0])

test['Age'] = test['Age'].fillna(round(np.mean(test['Age'])))
test['Fare'] = test['Fare'].fillna(round(np.mean(test['Fare'])))
test['Cabin'] = test['Cabin'].fillna('no value')

In [ ]:
print(train.isnull().sum())
print(test.isnull().sum())

# Y value


In [ ]:
print(train['Survived'].value_counts())
sns.countplot(x='Survived', data=train)

# X value

In [ ]:
train.columns

PassengerId is distinct, so delete

In [ ]:
print(train['PassengerId'].value_counts().value_counts())
print(test['PassengerId'].value_counts().value_counts())

train = train.drop(columns = 'PassengerId')
test = test.drop(columns = 'PassengerId')

Pclass

- ?? ???? ??? ?, Pclass? 1st ??? 2nd, 3rd?? ???? ?? ??? ???.
- ?? Pclass? ?, ???? ??. ? ???? ??? Ticket? ??? ??? ???.

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(12,5), constrained_layout=True)

sns.countplot(x="Pclass", data=train, hue='Survived', ax=axes[0]).set_title('train countplot')
sns.barplot(x="Pclass", y='Survived', data=train.groupby('Pclass').mean()['Survived'].reset_index(), ax=axes[1]).set_title('train survived_ratio')
sns.countplot(x="Pclass", data=test, ax=axes[2]).set_title('test Pclass')

Name

- ??? ???? ?? ??
- ??? ??? first name, title. last name?? ???? ??? ???.
- title? ???? ?? ???? ?????? ???? tilte? ????.
- ?? ??? ??? ?????? ??? ???. ??? ??? ? ?? ??? ??? ?? ???? ?? ??? ???? ???? ?? ???

In [ ]:
print(len(train['Name'])==len(np.unique(train['Name'])))

In [ ]:
# Name length column
train['Name_length'] = train['Name'].apply(len)
test['Name_length'] = test['Name'].apply(len)

- Pclass? Ticket class? ?????, ??? ??? Pclass? ??? ??? ??(?? ??? ??? ???)
- Fare ?? ??????? ??? ??? ? ? ???, 3rd? ???? ?? ?? ?? ?????? ??? ??? ?, ????? ????.

In [ ]:
fig, axes = plt.subplots(3,1, figsize=(12,6), constrained_layout=True)

sns.countplot(x="Name_length", data=train[train['Pclass']==1], ax=axes[0]).set_title('1st')
sns.countplot(x="Name_length", data=train[train['Pclass']==2], ax=axes[1]).set_title('2nd')
sns.countplot(x="Name_length", data=train[train['Pclass']==3], ax=axes[2]).set_title('3rd')

- ????? 1st, 2nd? 3rd?? ????? ?? ??? ?? ? ? ??.

extract title

In [ ]:
train['name_title'] = train['Name'].apply(lambda x: re.search('([A-Za-z]+)\.', x).group(1))
test['name_title'] = test['Name'].apply(lambda x: re.search('([A-Za-z]+)\.', x).group(1))

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(15,5), constrained_layout=True)

a = sns.countplot(x="name_title", data=train, order=train['name_title'].value_counts().index, ax=axes[0])
a.set_xticklabels(labels=train['name_title'].value_counts().index, rotation=45)
a.set_title('train name_title')
sns.countplot(x="name_title", data=test, order=test['name_title'].value_counts().index, ax=axes[1]).set_title('test name_title')

- ??? ?? ??? ??, ?? ???? ??? ?? ??? ?? etc? ????.

In [ ]:
train.loc[~train['name_title'].isin(test['name_title']), 'name_title'] = 'etc'
test.loc[~test['name_title'].isin(train['name_title']), 'name_title'] = 'etc'

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(15,5), constrained_layout=True)

sns.countplot(x="name_title", data=train, order=train['name_title'].value_counts().index, ax=axes[0]).set_title('train name_title')
sns.countplot(x="name_title", data=test, order=test['name_title'].value_counts().index, ax=axes[1]).set_title('test name_title')

Sex
- basemodel? ???? ?? ??? ?? ? ????.
- ??? ?? ?? ??? ?? ??, ??? ?? ????? ???? ??? ? ? ??.

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(12,5), constrained_layout=True)

sns.countplot(x="Sex", data=train, hue='Survived', ax=axes[0]).set_title('train countplot')
sns.barplot(x="Sex", y='Survived', data=train.groupby('Sex').mean()['Survived'].reset_index(), order=['male', 'female'], ax=axes[1]).set_title('train Sex_ratio')
sns.countplot(x="Sex", data=test, ax=axes[2]).set_title('test Pclass')

- Interaction effect? ???? ???..?

Age

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(8,4), constrained_layout=True)

for val in [0, 1]:
    sns.kdeplot(train.loc[train['Survived']==val, 'Age'], shade=True, label=val, ax=axes[0])

sns.kdeplot(data=test['Age'], shade=True, ax=axes[1])

- 20? ??? ?? ????? ?? ?? ? ? ??. ? ?? ???? ?? ??? ?? ??? ????? ??? ? ???? ??? ???.

In [ ]:
train['cate_age'] = pd.cut(train['Age'], [0, 15, 25, 40, 60, np.max(train['Age'])], labels=[1, 2, 3, 4, 5])
test['cate_age'] = pd.cut(test['Age'], [0, 15, 25, 40, 60, np.max(train['Age'])], labels=[1, 2, 3, 4, 5])

- train data? ??? ?, ? 4?? ???? ????, ? ???? ???? ?? bins??? ???.

sibsp and parch

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(8,6), constrained_layout=True)

sns.countplot(x='SibSp', data=train, hue='Survived', ax=axes[0])
sns.countplot(x='SibSp', data=test, ax=axes[1])

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(8,6), constrained_layout=True)

sns.countplot(x='Parch', data=train, hue='Survived', ax=axes[0])
sns.countplot(x='Parch', data=test, ax=axes[1])

- sibsp	# of siblings / spouses aboard the Titanic
- parch	# of parents / children aboard the Titanic
- ??? ? ??? ??? ??? ??? ??? ? ? ??.
- ??? ? ? ?? ??? Parch? SiSap? 0?? ?, ???? ?? ?? ? ? ??.

In [ ]:
train['family_size'] = train['SibSp'] + train['Parch']
test['family_size'] = test['SibSp'] + test['Parch']

- family_size??? ?? SibSp? Parch? ??? ? ??? ??? ???.

In [ ]:
crosstab_family_size = pd.crosstab(train['Ticket'], train['family_size'])
crosstab_family_size.loc[np.count_nonzero(crosstab_family_size==0, axis=1)!=8, ]

- ????? ?? family size? ???? ?? ??? ? ? ??.
- ????, family size? ???? ??? ????

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4), constrained_layout=True)

sns.countplot(x='family_size', data=train, hue='Survived', ax=axes[0])
sns.countplot(x='family_size', data=test, ax=axes[1])

- family_size? 1, 2, 3???? ???? ??.
- ???? [0, 4, 5, 6, 7, 10]? [1, 2, 3]? ?????.

In [ ]:
train.loc[train['family_size'].isin([0, 4, 5, 6, 7, 10]), 'family_size'] = 0
train.loc[train['family_size'].isin([1, 2, 3]), 'family_size'] = 1

In [ ]:
test.loc[test['family_size'].isin([0, 4, 5, 6, 7, 10]), 'family_size'] = 0
test.loc[test['family_size'].isin([1, 2, 3]), 'family_size'] = 1

Ticket
- ??? ????? ?????
- ??? ?? ? ????? ???, ????? ??? ? ???

In [ ]:
train.groupby('Ticket').mean()['Fare']

In [ ]:
train[['Ticket', 'Fare']]

?? ? ??? ??? ????.

In [ ]:
# Ticket length column
train['Ticket_length'] = train['Ticket'].apply(len)
test['Ticket_length'] = test['Ticket'].apply(len)

Cabin

In [ ]:
train['Cabin'].value_counts()

In [ ]:
train['Cabin_first'] = train['Cabin'].apply(lambda x: x[0])
test['Cabin_first'] = test['Cabin'].apply(lambda x: x[0])

- ??? ????? ??? ? ????.

embarked

fig, axes = plt.subplots(1,3, figsize=(8,6), constrained_layout=True)

sns.countplot(x='Embarked', data=train, hue='Survived', ax=axes[0])
sns.barplot(x='Embarked', y='Survived', data=train.groupby('Embarked').mean()['Survived'].reset_index(), order=['S', 'C', 'Q'], ax=axes[1])
sns.countplot(x='Embarked', data=test, order=['S', 'C', 'Q'], ax=axes[2])

survived ratio

a = train.groupby(['cate_age', 'Pclass', 'Sex']).mean()['Survived']

a = a.fillna(0).reset_index()

a.columns = ['cate_age', 'Pclass', 'Sex', 'est_Survived']

train = pd.merge(train, a, how='left', on=['cate_age', 'Pclass', 'Sex'])
test = pd.merge(test, a, how='left', on=['cate_age', 'Pclass', 'Sex'])

- target column? ????, ????? ?????? ?, ???? ???? ????, ????? ???? ???? ?? ? ? ??.

---

# features

features = ['Name_length', 'est_Survived', 'Pclass', 'Sex', 'Embarked', 'na_count', 'name_title', 'cate_age', 'family_size']

train2 = train[features]
test2 = test[features]

# lebeling

label = LabelEncoder()

df = pd.concat([train2, test2])

for i in ['Sex', 'Embarked', 'name_title']:
    df[i] = label.fit_transform(df[i])

train2 = df.iloc[:891, :]
test2 = df.iloc[891:, :]

# one hot encoding

train2 = pd.concat([train['Survived'], train2.iloc[:, :2], pd.get_dummies(train2.iloc[:, 2:].astype(object))], axis=1)

test2 = pd.concat([test2.iloc[:, :2], pd.get_dummies(test2.iloc[:, 2:].astype(object))], axis=1)

# Modeling

## Random Forest

def rf_model(train_val, test_val):
    regr = RandomForestRegressor(max_depth=max_depth, random_state=0, n_estimators=100).fit(
    train_val.drop(columns=['Survived']), train_val['Survived'])
    
    test_val_y = regr.predict(test_val.drop(columns=['Survived'])) > decision_boundary
    test_val_y = test_val_y.astype(int)
    
    true_count = 0
    for i in test_val_y - test_val['Survived']:
        if i == 0:
            true_count+=1
    
    return true_count/len(test_val_y)

# 5-fold

cv = KFold(5, shuffle=True, random_state=0)

for j in range(2, 9):
    for k in range(30, 51):
        L = []
        for i, (idx_train, idx_test) in enumerate(cv.split(train2)):
            train_val = train2.iloc[idx_train]
            test_val = train2.iloc[idx_test]
            
            max_depth=j
            decision_boundary=k/100
            
            L.append(rf_model(train_val, test_val))
            if np.mean(L) > .83:
                print('accuacy : {}, max_depth : {}, decision boundary : {}'.format(np.mean(L), j, k/100))


## Xgboost

## 5-fold CV

cv = KFold(5, shuffle=True, random_state=0)
L = []
for i, (idx_train, idx_test) in enumerate(cv.split(train)):
    train_val = train.iloc[idx_train]
    test_val = train.iloc[idx_test]

    L.append(xgb_model(train_val, test_val))
    
print(np.mean(L)) 

## Save Submission

### Random Forest

regr = RandomForestRegressor(max_depth=4, random_state=0, n_estimators=100).fit(
    train2[test2.columns], train2['Survived'])
test_y = regr.predict(test2)
test_y = test_y > .5
predictions = test_y.astype(int)
test_PassengerId = pd.read_csv('../input/gender_submission.csv')['PassengerId']
submission = pd.concat([pd.DataFrame(test_PassengerId), pd.DataFrame({'Survived':predictions})], axis=1)
submission.to_csv("submission_rf4.csv", index=False)

### Xgboost

### ensembles